In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ==========================================
# 1. GENERATE THE RAW MULTI-TENANT LOG FILE
# ==========================================
np.random.seed(42)
num_records = 1000
tenants = ['Acme_Corp', 'Beta_Industries', 'Delta_Systems', 'Omega_Labs']
tiers = ['Basic', 'Professional', 'Enterprise']
statuses = ['Active', 'Active', 'Active', 'Churned']

generated_data = []
start_date = datetime(2025, 1, 1)

for i in range(1, num_records + 1):
    tenant = np.random.choice(tenants)
    tier = np.random.choice(tiers)
    status = np.random.choice(statuses)
    
    monthly_price = 49.00 if tier == 'Basic' else (149.00 if tier == 'Professional' else 499.00)
    days_offset = np.random.randint(0, 500)
    signup_date = start_date + timedelta(days=days_offset)
    api_calls = np.random.randint(500, 100000) if status == 'Active' else np.random.randint(10, 5000)
    storage_used_gb = np.random.uniform(1.5, 500.0) if tier != 'Basic' else np.random.uniform(0.1, 50.0)
    
    generated_data.append({
        "account_id": f"ACC_{10000 + i}",
        "tenant_id": tenant,
        "subscription_tier": tier,
        "monthly_recurring_revenue": monthly_price,
        "account_status": status,
        "api_requests_count": api_calls,
        "cloud_storage_used_gb": round(storage_used_gb, 2),
        "activation_date": signup_date.strftime("%Y-%m-%d")
    })

raw_saas_df = pd.DataFrame(generated_data)
raw_csv_path = "raw_saas_tenant_logs.csv"
raw_saas_df.to_csv(raw_csv_path, index=False)
print(f"💾 File saved successfully as: '{raw_csv_path}'")

# ==========================================
# 2. INGEST DATA AND ROUTE SECURELY
# ==========================================
class TenantDataRouter:
    def __init__(self, file_path):
        self.data = pd.read_csv(file_path)
        
    def get_tenant_data(self, requesting_tenant_id):
        # Strict isolation filter
        isolated_data = self.data[self.data['tenant_id'] == requesting_tenant_id]
        
        # Security leak check
        leaked_rows = isolated_data[isolated_data['tenant_id'] != requesting_tenant_id]
        if not leaked_rows.empty:
            raise SecurityError("CRITICAL: Cross-Tenant Data Isolation Breach!")
            
        return isolated_data

# Initialize router with our file
router = TenantDataRouter(raw_csv_path)

# Test execution for Beta Industries
beta_isolated_data = router.get_tenant_data("Beta_Industries")
print(f"🔒 Beta_Industries isolation active. Row count fetched: {len(beta_isolated_data)}")